# Orchestration Fundamentals

**Module:** 14 — AI Orchestration

Coordinate models, tools, humans, and data across reliable multi-step AI systems.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define orchestration vs a single LLM call
- Contrast chains, workflows, agents, and platforms
- Separate control plane vs data plane concerns
- List reliability requirements (retries, timeouts, idempotency, state)


## What Is AI Orchestration?

### Definition
Orchestration is the control logic that sequences and coordinates models, tools, data fetches, and humans so a multi-step goal completes reliably.

### Why it matters
Real products are pipelines and graphs, not one prompt. Without orchestration you get brittle scripts: no resume, no budgets, no approvals, no observability.

### How it works
A runtime interprets a plan (fixed chain, DAG, or agent-chosen actions), persists state, applies policies (timeouts, rate limits, PII), and emits traces.

### Intuition
The conductor of an orchestra: players (models/tools) are skilled, but someone must cue entrances, cutoffs, and recoveries.

### Pitfalls
- Embedding all control flow only inside prompts
- No durable state → crash = restart from zero
- Treating non-idempotent tools as safe to retry blindly

### When to use
Multi-tool research, document pipelines, HITL approvals, batch generation, long-running jobs.


### Chains vs Workflows vs Agents vs Platforms

| Style | Who decides next step? | Flexibility | Best for |
|-------|------------------------|-------------|----------|
| **Chain** | Developer (linear) | Low | Stable ETL-like LLM steps |
| **Workflow / DAG** | Developer (graph) | Medium | Branching business processes |
| **Agent** | Model (tool loop) | High | Open-ended tasks |
| **Platform** | Mix + ops tooling | Varies | Team-scale production |

```mermaid
flowchart TB
  U[User Goal] --> CP[Control Plane]
  CP --> W[Workflow / Agent Runtime]
  W --> M[Models]
  W --> T[Tools]
  W --> H[Humans]
  W --> S[(State / Checkpoints)]
  M --> DP[Data Plane: tokens, docs, embeddings]
  T --> DP
```


In [ ]:
# Demo 1: fixed chain orchestrator
from typing import Callable
Step = Callable[[dict], dict]

def step_classify(ctx: dict) -> dict:
    text = ctx["input"].lower()
    ctx["intent"] = "billing" if "invoice" in text else "general"
    return ctx

def step_route(ctx: dict) -> dict:
    ctx["handler"] = "billing_agent" if ctx["intent"] == "billing" else "faq_agent"
    return ctx

def step_respond(ctx: dict) -> dict:
    ctx["output"] = f"Routed to {ctx['handler']} for: {ctx['input']}"
    return ctx

def run_chain(pipeline: list[Step], ctx: dict) -> dict:
    for step in pipeline:
        ctx = step(ctx)
        ctx.setdefault("trace", []).append(step.__name__)
    return ctx

result = run_chain(
    [step_classify, step_route, step_respond],
    {"input": "Where is my invoice?"},
)
print(result)


In [ ]:
# Demo 2: DAG with parallel fan-out then join
from concurrent.futures import ThreadPoolExecutor

def extract_entities(text: str) -> list[str]:
    return [w.strip(",.") for w in text.split() if w.istitle()]

def sentiment(text: str) -> str:
    return "neg" if any(x in text.lower() for x in ("angry", "broken", "refund")) else "neu"

def risk(text: str) -> str:
    return "high" if "lawyer" in text.lower() else "low"

def run_dag(text: str) -> dict:
    with ThreadPoolExecutor(max_workers=3) as ex:
        f1, f2, f3 = ex.submit(extract_entities, text), ex.submit(sentiment, text), ex.submit(risk, text)
        return {"entities": f1.result(), "sentiment": f2.result(), "risk": f3.result()}

print(run_dag("Angry customer Ada wants a refund and mentioned a Lawyer"))


In [ ]:
# Demo 3: agent-style loop (model chooser mocked)
TOOLS = {
    "search": lambda q: f"results-for:{q}",
    "calc": lambda e: str(eval(e, {"__builtins__": {}}, {})),  # toy only
}

def agent_loop(goal: str, max_steps=4):
    scratch = {"goal": goal, "obs": []}
    plan = [("search", "invoice 99"), ("calc", "19.99*1.2"), ("finish", "done")]
    for i, (tool, arg) in enumerate(plan[:max_steps]):
        if tool == "finish":
            scratch["answer"] = f"Completed after {i} tools; obs={scratch['obs']}"
            return scratch
        scratch["obs"].append({tool: TOOLS[tool](arg)})
    scratch["answer"] = "max steps"
    return scratch

print(agent_loop("Estimate tax on invoice"))


## Control Plane vs Data Plane

### Definition
The **control plane** decides *what may run* (routing, authz, budgets, retries). The **data plane** moves *payloads* (tokens, embeddings, files, tool I/O).

### Why it matters
Security and cost bugs appear when policy lives only in prompts. Control-plane enforcement is auditable and consistent.

### How it works
Put rate limits, PII redaction flags, model allow-lists, and approval gates in control logic. Keep prompts focused on task content.

### Intuition
Air traffic control (control) vs the airplanes and cargo (data).

### Pitfalls
- Prompt-only 'do not call delete' with a delete tool still attached
- Logging raw data-plane PII in control traces

### When to use
Always — even small apps benefit from a thin control layer.


In [ ]:
# Demo 4: tiny control plane policy gate
from dataclasses import dataclass

@dataclass
class Policy:
    max_tool_calls: int = 5
    allow_tools: set[str] | None = None
    max_prompt_chars: int = 4000

def authorize(tool: str, prompt: str, tool_calls_so_far: int, policy: Policy) -> str | None:
    if tool_calls_so_far >= policy.max_tool_calls:
        return "deny: tool budget"
    if policy.allow_tools is not None and tool not in policy.allow_tools:
        return "deny: tool not allow-listed"
    if len(prompt) > policy.max_prompt_chars:
        return "deny: prompt too large"
    return None

p = Policy(allow_tools={"search", "calc"})
for tool in ["search", "delete_db", "calc"]:
    print(tool, authorize(tool, "x"*10, 1, p))


## Reliability Requirements Checklist

| Concern | Question | Starter tactic |
|---------|----------|----------------|
| Retries | Which errors are transient? | Exponential backoff + jitter |
| Idempotency | Can we safely retry? | Idempotency keys |
| Timeouts | What's the budget? | Per-step + global deadlines |
| State | Can we resume? | Checkpoints |
| Compensations | How to undo? | Saga / compensating actions |
| Budgets | Token/$ caps? | Control-plane meters |
| Observability | Can we debug? | Trace ids per run |

### When you need orchestration
- Multi-tool research agents
- Document processing pipelines
- Human-approval loops
- Long-running jobs with resume
- Fan-out evaluation / batch generation


In [ ]:
# Demo 5: retry with backoff (educational)
import time, random

def flaky(n=[0]):
    n[0] += 1
    if n[0] < 3:
        raise TimeoutError("transient")
    return "ok"

def retry(fn, attempts=5, base=0.01):
    for i in range(attempts):
        try:
            return fn()
        except TimeoutError as e:
            sleep = base * (2 ** i) + random.random() * base
            time.sleep(sleep)
            last = e
    raise last

print(retry(flaky))


### Try it yourself — Orchestration basics

1. Add a global timeout deadline to `run_chain` (fail if wall clock exceeds N ms).
2. Extend the policy gate to deny when `risk=high` unless `human_approved=True` in ctx.
3. Draw your product's control vs data plane in ASCII.

**Stretch:** Make the agent loop choose tools via a scored heuristic instead of a fixed plan.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `chain` | Linear sequence of steps |
| `DAG` | Directed acyclic graph of dependencies |
| `control plane` | Policy, routing, budgets, permissions |
| `data plane` | Model/tool payloads and content |
| `idempotency` | Same request can be repeated safely |


## Worked Scenario — Invoice Dispute

```mermaid
sequenceDiagram
  participant U as User
  participant O as Orchestrator
  participant M as Model
  participant T as Tools
  participant H as Human
  U->>O: Dispute invoice
  O->>T: fetch_invoice
  O->>M: classify + draft
  alt amount > 100 or risk high
    O->>H: approve
    H->>O: decision
  end
  O->>T: open_case
  O->>U: response
```

Control plane enforces: tool allow-list, amount threshold, timeout 20s, budget $0.05.


In [ ]:
# Map styles to the scenario
decision_table = [
    ("stable sanitize→classify→template", "chain"),
    ("branch on risk + HITL", "workflow"),
    ("open-ended investigate across 10 tools", "agent"),
]
for need, style in decision_table:
    print(f"{style:10} | {need}")


In [ ]:
# Reliability checklist scorer
checks = {
    "retries": True,
    "idempotency_keys": True,
    "timeouts": True,
    "checkpoints": False,
    "traces": True,
    "budgets": False,
}
score = sum(checks.values()) / len(checks)
print("readiness", round(score, 2), "missing", [k for k, v in checks.items() if not v])


### Try it yourself — Fundamentals deepen

1. Rewrite Demo 1 chain to emit OpenTelemetry-like dict spans.
2. Add compensations: if `open_case` fails after email send, enqueue 'notify_ops'.


## Glossary Drill
Explain each in one sentence without notes: chain, DAG, control plane, data plane, idempotency, compensation, orchestration vs choreography.


## Key Takeaways

- Orchestration = reliability + coordination for AI systems
- Pick chain vs workflow vs agent based on task variability
- Persist state early; separate control policies from model I/O
- Retries without idempotency are incidents waiting to happen
